# Baitarani: CWC stations + India-WRIS water level and discharge

**Settings:** Accelerator **None**, Internet **On**. Then **Save Version → Save & Run All (Commit)**, or run all cells. Every cell keeps going after an error, so the run always finishes.

Steps:
1. Install HydroSwift (from GitHub; it is not on PyPI).
2. CWC station list for the Baitarani (warning / danger / HFL levels).
3. India-WRIS: past daily **water level** and **discharge** for Anandpur, Akhuapada, Champua and Swampatna (2000–2024).
4. Summary: period covered, number of days, and days that have both level and flow (needed for the rating curve).

Results go to the **Output** tab: `results/baitarani/` plus `baitarani_results.zip`.

In [ ]:
import os, subprocess, glob, traceback
import pandas as pd
pd.set_option('display.width', 200)
OUT = '/kaggle/working/results/baitarani'
os.makedirs(OUT, exist_ok=True)
NAMES = 'anandpur|akhuapada|champua|swampatna|keshdihi|bhejdih'
print(subprocess.run('pip install -q git+https://github.com/carbform/HydroSwift.git', shell=True,
                     capture_output=True, text=True).stderr[-1500:])
import hydroswift
print('HydroSwift', getattr(hydroswift, '__version__', '?'))
try:
    hydroswift.help()
except Exception as e:
    print('help() not available:', e)

## 2. CWC flood-forecasting stations on the Baitarani

In [ ]:
cwc = None
for kw in [dict(river='Baitarani'), dict(river='Baitarni'), {}]:
    try:
        s = hydroswift.cwc.stations(**kw)
        if not kw:
            txt = s.astype(str).apply(lambda c: c.str.lower())
            s = s[txt.apply(lambda c: c.str.contains(NAMES)).any(axis=1)]
        print(kw or 'all stations filtered by name', '->', len(s))
        if len(s):
            cwc = s
            break
    except Exception as e:
        print(kw, 'failed:', e)
if cwc is not None:
    print(cwc.to_string())
    cwc.to_csv(f'{OUT}/cwc_baitarani_stations.csv', index=False)

## 3. India-WRIS: which Baitarani stations have water level / discharge?

In [ ]:
for f in ['variables', 'basins']:
    try:
        r = getattr(hydroswift.wris, f)()
        print(f'--- wris.{f}()'); print(r.to_string()[:3000] if hasattr(r, 'to_string') else r)
    except Exception as e:
        print(f'wris.{f}() failed:', e)

found = {}
VARS = ['discharge', 'water_level', 'water level', 'level', 'stage']
BASINS = ['Brahmani and Baitarni', 'Brahmani and Baitarani', 'Brahmani-Baitarani', 'Brahmani Baitarani', 'Brahmani']
for var in VARS:
    for basin in BASINS:
        try:
            s = hydroswift.wris.stations(basin=basin, variable=var)
        except Exception as e:
            print(f'{var!r} / {basin!r}: {str(e)[:150]}')
            continue
        txt = s.astype(str).apply(lambda c: c.str.lower())
        hit = s[txt.apply(lambda c: c.str.contains(NAMES)).any(axis=1)]
        print(f'{var!r} / {basin!r}: {len(s)} stations in basin, {len(hit)} on our list')
        if len(hit):
            key = 'level' if var != 'discharge' else 'discharge'
            if key not in found:
                found[key] = hit
                print(hit.to_string()[:3000])
                hit.to_csv(f'{OUT}/wris_stations_{key}.csv', index=False)
            break
print('\nFound:', {k: len(v) for k, v in found.items()})

## 4. Download 2000–2024 (stations without data are skipped)

In [ ]:
data = {}
for key, s in found.items():
    d = f'{OUT}/wris_{key}'
    try:
        r = hydroswift.fetch(s, start_date='2000-01-01', end_date='2024-12-31', output_dir=d, merge=True)
        print(key, '->', type(r).__name__)
        if hasattr(r, 'head'):
            data[key] = r
            print(r.head(5)); print('rows:', len(r))
    except Exception as e:
        print(key, 'download failed:', e)
        traceback.print_exc(limit=2)
print(subprocess.run(f'find {OUT} -type f | head -100; du -sh {OUT}', shell=True, capture_output=True, text=True).stdout)

## 5. Summary: what period does each file cover?

In [ ]:
rows = []
for p in sorted(glob.glob(f'{OUT}/**/*', recursive=True)):
    if not p.lower().endswith(('.csv', '.xlsx', '.parquet')) or 'stations' in os.path.basename(p):
        continue
    try:
        df = pd.read_parquet(p) if p.endswith('.parquet') else (pd.read_excel(p) if p.endswith('.xlsx') else pd.read_csv(p, low_memory=False))
    except Exception as e:
        print('cannot read', p, e); continue
    tcol = next((c for c in df.columns if any(k in str(c).lower() for k in ('date', 'time'))), None)
    if tcol is None:
        continue
    t = pd.to_datetime(df[tcol], errors='coerce', format='mixed')
    num = df.select_dtypes('number').columns.tolist()
    rows.append(dict(file=os.path.relpath(p, OUT), rows=len(df), start=t.min(), end=t.max(),
                     step=str(t.sort_values().diff().median()), days=t.dt.normalize().nunique(), columns=num[:5]))
s = pd.DataFrame(rows)
print(s.to_string() if len(s) else 'No time series files were downloaded.')
s.to_csv(f'{OUT}/files_summary.csv', index=False)

# days with BOTH water level and discharge per station -> needed for the rating curve
if 'level' in data and 'discharge' in data:
    for nm in ['anandpur', 'akhuapada', 'champua', 'swampatna']:
        def pick(df):
            txt = df.astype(str).apply(lambda c: c.str.lower())
            m = txt.apply(lambda c: c.str.contains(nm)).any(axis=1)
            return df[m] if m.any() else None
        a, b = pick(data['level']), pick(data['discharge'])
        if a is None or b is None:
            print(nm, ': level', a is not None, '/ discharge', b is not None); continue
        ta = next(c for c in a.columns if 'date' in str(c).lower() or 'time' in str(c).lower())
        tb = next(c for c in b.columns if 'date' in str(c).lower() or 'time' in str(c).lower())
        da = set(pd.to_datetime(a[ta], errors='coerce').dt.normalize().dropna())
        db = set(pd.to_datetime(b[tb], errors='coerce').dt.normalize().dropna())
        print(f'{nm}: level days {len(da)}, discharge days {len(db)}, both {len(da & db)}')

In [ ]:
print(subprocess.run('cd /kaggle/working && zip -qr baitarani_results.zip results/baitarani && ls -lh baitarani_results.zip',
                     shell=True, capture_output=True, text=True).stdout)